### Đọc file busstop và khu điểm du lịch

In [5]:
import pandas as pd
import json

busstopPath = 'data/transportation/stops_hcm.json'
diemDuLichPath = 'data/DiaDiemDuLich/KhuDiemDuLich.xlsx'

# 1. Đọc file Excel các địa điểm du lịch
df_locations = pd.read_excel(diemDuLichPath)
print("--- Dữ liệu các địa điểm du lịch ---")
print(df_locations.head())

# 2. Đọc file JSON các trạm xe buýt
with open(busstopPath, 'r', encoding='utf-8') as f:
    data_stops = json.load(f)
df_stops = pd.DataFrame(data_stops)
print("\n--- Dữ liệu các trạm xe buýt ---")
print(df_stops.head())

--- Dữ liệu các địa điểm du lịch ---
      mdn_mst                  tenKhuDiemDuLich  \
0  0302873402   Bảo tàng Lịch sử TP.Hồ Chí Minh   
1  0302713550  Bảo tàng Mỹ thuật TP.Hồ Chí Minh   
2  0302945625   Bảo tàng Chứng tích Chiến tranh   
3  0302790403    Bảo tàng Thành phố Hồ Chí Minh   
4  0309511680            Bảo tàng Phụ nữ Nam bộ   

                                              diaChi          moTa  \
0             2 Nguyễn Bỉnh Khiêm, Bến Nghé, Quận 1   Điểm du lịch   
1  97A Phó Đức Chính, Phường Nguyễn Thái Bình, Qu...  Điểm du lịch   
2                    28 Võ Văn Tần, Phường 6, Quận 3  Điểm du lịch   
3                   65 Lý Tự Trọng, Bến Nghé, Quận 1  Điểm du lịch   
4                   202 Võ Thị Sáu, Phường 7, Quận 3  Điểm du lịch   

  donViChuTriCungCap ngayCungCap tanSuatCungCap  ghiChu  
0   UBNDTP công nhận  24/02/2025            Năm     NaN  
1   UBNDTP công nhận  24/02/2025            Năm     NaN  
2   UBNDTP công nhận  24/02/2025            Năm     NaN  
3  

### Xử lý file PDF và culture

In [7]:
from langchain.document_loaders import PyPDFLoader
culturePath = 'data/culture/culture_vi_enriched.json'
vanHoavaDuLichPath = 'data/VanHoaVaDuLichVN.pdf'

# 1. Đọc và lọc file JSON về văn hóa
with open(culturePath, 'r', encoding='utf-8') as f:
    data_culture = json.load(f)

# Lọc ra các bài viết chỉ liên quan đến TP.HCM
hcm_keywords = ['Hồ Chí Minh', 'Sài Gòn', 'TP.HCM']
hcm_culture_docs = []
for item in data_culture:
    # Kiểm tra xem tiêu đề hoặc nội dung có chứa từ khóa không
    if any(keyword in item['title'] for keyword in hcm_keywords) or any(keyword in item['content'] for keyword in hcm_keywords):
        # Tạo một đối tượng Document mà LangChain có thể hiểu
        from langchain.docstore.document import Document
        doc = Document(page_content=item['content'], metadata={'source': item['source'], 'title': item['title']})
        hcm_culture_docs.append(doc)

print(f"\n--- Tìm thấy {len(hcm_culture_docs)} tài liệu văn hóa liên quan đến TP.HCM từ file JSON ---")

# 2. Đọc và lọc file PDF
pdf_loader = PyPDFLoader(vanHoavaDuLichPath)
pdf_pages = pdf_loader.load()

hcm_pdf_docs = []
for page in pdf_pages:
    # Lọc các trang có chứa từ khóa về TP.HCM
    if any(keyword in page.page_content for keyword in hcm_keywords):
        hcm_pdf_docs.append(page)

print(f"--- Tìm thấy {len(hcm_pdf_docs)} trang PDF liên quan đến TP.HCM ---")

# Tổng hợp tất cả tài liệu dạng văn bản
all_text_documents = hcm_culture_docs + hcm_pdf_docs
print(f"==> Tổng cộng có {len(all_text_documents)} tài liệu văn bản để đưa vào Vector DB.")

c:\Users\tuand\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Disabling PyTorch because PyTorch >= 2.1 is required but found 2.0.1+cu117



--- Tìm thấy 6 tài liệu văn hóa liên quan đến TP.HCM từ file JSON ---
--- Tìm thấy 95 trang PDF liên quan đến TP.HCM ---
==> Tổng cộng có 101 tài liệu văn bản để đưa vào Vector DB.


In [6]:
import os
import re
import json
import unicodedata
import hashlib
import argparse
from pathlib import Path
from bs4 import BeautifulSoup
from tqdm import tqdm

# optional pdf text extraction
try:
    from pdfminer.high_level import extract_text as extract_pdf_text
except Exception:
    extract_pdf_text = None

import pandas as pd

# --------- Config mặc định (sửa khi cần) ----------
DEFAULT_INPUT = {
    "culture_enriched": "data/culture/culture_vi_enriched.json",
    "coso": "data/DiaDiemDuLich/CoSoDatChuan.json",
    "khu_diem": "data/DiaDiemDuLich/KhuDiemDuLich.xlsx",
    "stops": "data/transportation/stops_hcm.json",
    "pdf_vanhoa": "data/VanHoaVaDuLichVN.pdf",
}
OUT_DIR = Path("processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_JSONL = OUT_DIR / "dataset_clean.jsonl"
OUT_JSON = OUT_DIR / "dataset_clean.json"
# ngưỡng ký tự tối thiểu để giữ record
MIN_CONTENT_LEN = 100
# cho domain 'giao_thong' cho phép ngắn hơn
MIN_CONTENT_LEN_SHORT_DOMAIN = 30

# ---------- helper functions ----------
def normalize_text(s):
    if s is None:
        return ""
    if not isinstance(s, str):
        s = str(s)
    # Unicode NFC
    s = unicodedata.normalize("NFC", s)
    # remove HTML tags
    s = BeautifulSoup(s, "html.parser").get_text(" ", strip=True)
    # remove control chars
    s = re.sub(r'[\x00-\x1f\x7f]+', ' ', s)
    # collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def hash_text(s):
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

def safe_load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        try:
            return json.load(f)
        except Exception:
            f.seek(0)
            return [json.loads(line) for line in f if line.strip()]

# ---------- loaders ----------
def load_culture(path):
    out = []
    data = safe_load_json(path)
    for i, item in enumerate(data):
        title = normalize_text(item.get("title") or item.get("name") or "")
        content = normalize_text(item.get("content") or item.get("description") or item.get("summary") or "")
        url = item.get("source") or item.get("url") or item.get("link") or ""
        out.append({
            "id": f"culture_{i+1}",
            "domain": "van_hoa",
            "title": title,
            "content": content,
            "source_url": url,
            "metadata": item
        })
    return out

def load_coso(path):
    out = []
    data = safe_load_json(path)
    for i, item in enumerate(data):
        # attempt common keys in Vietnamese or snake_case
        name = normalize_text(item.get("Tên cơ sở") or item.get("Tên cơ sở".lower()) or item.get("ten_co_so") or item.get("name") or "")
        desc = normalize_text(item.get("Mô tả") or item.get("mo_ta") or item.get("description") or "")
        address = normalize_text(item.get("Địa chỉ") or item.get("dia_chi") or item.get("address") or "")
        provider = normalize_text(item.get("Đơn vị cung cấp") or item.get("don_vi") or "")
        freq = normalize_text(item.get("tần suất cung cấp") or item.get("tan_suat") or "")
        content = " ".join([x for x in (desc, address) if x])
        out.append({
            "id": f"coso_{i+1}",
            "domain": "du_lich",
            "title": name,
            "content": content,
            "source_url": item.get("source") or item.get("url") or "",
            "metadata": item
        })
    return out

def load_khudiem(path):
    out = []
    try:
        xls = pd.read_excel(path, sheet_name=None)
    except Exception:
        xls = {"sheet1": pd.read_excel(path, sheet_name=0)}
    idx = 1
    for sheet_name, df in xls.items():
        df = df.fillna("")
        for _, row in df.iterrows():
            title = normalize_text(row.get("Name") or row.get("Tên") or row.get("title") or row.get("name") or "")
            address = normalize_text(row.get("Address") or row.get("Địa chỉ") or row.get("diachi") or "")
            desc = normalize_text(row.get("Description") or row.get("Mô tả") or row.get("Ghi chú") or "")
            content = " ".join([x for x in (desc, address) if x])
            out.append({
                "id": f"khu_{idx}",
                "domain": "du_lich",
                "title": title,
                "content": content,
                "source_url": "",
                "metadata": dict(row)
            })
            idx += 1
    return out

def load_stops(path):
    out = []
    data = safe_load_json(path)
    for item in data:
        name = normalize_text(item.get("Name") or item.get("name") or item.get("Tên") or "")
        street = normalize_text(item.get("Street") or item.get("street") or "")
        zone = normalize_text(item.get("Zone") or item.get("zone") or "")
        routes = normalize_text(item.get("Routes") or item.get("routes") or "")
        addr = " ".join([x for x in (str(item.get("AddressNo","")), street, zone) if x])
        content = f"Trạm dừng: {name}. Địa chỉ: {addr}. Các tuyến: {routes}"
        out.append({
            "id": f"stop_{item.get('StopId') or item.get('id') or ''}",
            "domain": "giao_thong",
            "title": name,
            "content": normalize_text(content),
            "source_url": item.get("source") or "",
            "metadata": item
        })
    return out

def load_pdf(path, max_parts=200):
    out = []
    if extract_pdf_text is None:
        # no pdf extraction library
        out.append({
            "id": "pdf_meta",
            "domain": "van_hoa",
            "title": Path(path).name,
            "content": "",
            "source_url": str(path),
            "metadata": {"note": "pdfminer_not_installed"}
        })
        return out
    try:
        txt = extract_pdf_text(path)
        txt = normalize_text(txt)
        # split into paragraphs by double newline
        parts = [p.strip() for p in re.split(r'\n{2,}', txt) if p.strip()]
        for i, part in enumerate(parts[:max_parts]):
            out.append({
                "id": f"pdf_part_{i+1}",
                "domain": "van_hoa",
                "title": (part[:80] + '...') if len(part) > 0 else Path(path).name,
                "content": part,
                "source_url": str(path),
                "metadata": {"page_chunk": i+1}
            })
        return out
    except Exception as e:
        out.append({
            "id": "pdf_meta",
            "domain": "van_hoa",
            "title": Path(path).name,
            "content": "",
            "source_url": str(path),
            "metadata": {"note": "pdf_extract_failed", "error": str(e)}
        })
        return out

# ---------- main processing ----------
def process_all(inputs, min_len=MIN_CONTENT_LEN, min_len_short=MIN_CONTENT_LEN_SHORT_DOMAIN):
    records = []
    # load each present source
    if Path(inputs.get("culture_enriched")).exists():
        records += load_culture(inputs.get("culture_enriched"))
    if Path(inputs.get("coso")).exists():
        records += load_coso(inputs.get("coso"))
    if Path(inputs.get("khu_diem")).exists():
        records += load_khudiem(inputs.get("khu_diem"))
    if Path(inputs.get("stops")).exists():
        records += load_stops(inputs.get("stops"))
    if Path(inputs.get("pdf_vanhoa")).exists():
        records += load_pdf(inputs.get("pdf_vanhoa"))

    # deduplicate and filter
    seen = set()
    cleaned = []
    for r in records:
        title = (r.get("title") or "").strip()
        content = (r.get("content") or "").strip()
        if not title and len(content) < min_len:
            continue
        key = (title + "|" + content[:300])
        h = hash_text(key)
        if h in seen:
            continue
        seen.add(h)
        # length filter
        if len(content) < min_len:
            if r.get("domain") == "giao_thong" and len(content) >= min_len_short:
                cleaned.append(r)
            else:
                continue
        else:
            cleaned.append(r)

    # save jsonl
    with open(OUT_JSONL, "w", encoding="utf-8") as fo:
        for r in cleaned:
            fo.write(json.dumps(r, ensure_ascii=False) + "\n")
    # save json array
    with open(OUT_JSON, "w", encoding="utf-8") as fo:
        json.dump(cleaned, fo, ensure_ascii=False, indent=2)

    return {
        "raw_count": len(records),
        "cleaned_count": len(cleaned),
        "out_jsonl": str(OUT_JSONL),
        "out_json": str(OUT_JSON)
    }

# ---------- CLI ----------
if __name__ == "__main__":
    import sys
    parser = argparse.ArgumentParser(description="Stage1 preprocessing: normalize, dedupe, filter")
    parser.add_argument("--input_culture", default=DEFAULT_INPUT["culture_enriched"])
    parser.add_argument("--input_coso", default=DEFAULT_INPUT["coso"])
    parser.add_argument("--input_khudiem", default=DEFAULT_INPUT["khu_diem"])
    parser.add_argument("--input_stops", default=DEFAULT_INPUT["stops"])
    parser.add_argument("--input_pdf", default=DEFAULT_INPUT["pdf_vanhoa"])
    parser.add_argument("--min_len", type=int, default=MIN_CONTENT_LEN, help="min content length to keep")
    parser.add_argument("--min_len_short", type=int, default=MIN_CONTENT_LEN_SHORT_DOMAIN, help="min len for short-domain like stops")

    # parse_known_args() sẽ bỏ qua các tham số lạ (ví dụ --f=... của Jupyter)
    args, unknown = parser.parse_known_args()
    inputs = {
        "culture_enriched": args.input_culture,
        "coso": args.input_coso,
        "khu_diem": args.input_khudiem,
        "stops": args.input_stops,
        "pdf_vanhoa": args.input_pdf
    }
    res = process_all(inputs, min_len=args.min_len, min_len_short=args.min_len_short)
    print("Done. summary:", res)


Done. summary: {'raw_count': 4466, 'cleaned_count': 4297, 'out_jsonl': 'processed\\dataset_clean.jsonl', 'out_json': 'processed\\dataset_clean.json'}
